'''
Source: Impact4cast (Max Planck Institute)
Original code from the Max Planck Institute for Informatics / Max Planck Institute for Security and Privacy research group.

Modifications: Bug fixes for checkpoint resume mechanism and dataset adaptation for scientific entity impact prediction.
'''


将分散在多个gz文件中的概念共现边数据进行合并整合，它遍历读取concept_pair文件夹中的所有分块边数据文件，跳过空文件并将所有边记录合并到一个完整的列表中，最终将整合后的全部概念关联边保存为单个的all_concept_pairs.gz文件，同时记录整个合并过程的详细日志。

In [ ]:
import gzip
import os
import time
from datetime import datetime
import pickle
import gc
import sys
import re

# ========== 配置区域 ==========
SOURCE_FOLDER = r"H:\ASIST\entities_pair"
OUTPUT_FILE =r"G:\output\merged_edges.pkl.gz" 
LOG_FOLDER = r"H:\ASIST\logs"

# 处理参数
BATCH_SIZE = 100  # 每批处理的文件数量（减少因为gz文件可能更大）
MAX_EDGES_PER_CHUNK = 50000  # 每个临时文件的最大边数
PICKLE_PROTOCOL = pickle.HIGHEST_PROTOCOL  # 统一使用最高协议

# ========== 初始化 ==========
if not os.path.exists(LOG_FOLDER):
    os.makedirs(LOG_FOLDER)

log_file = os.path.join(LOG_FOLDER, f'log_merge_concept_pairs_{datetime.now().strftime("%Y%m%d_%H%M%S")}.txt')

def log_message(message, to_console=True, to_file=True):
    """记录日志"""
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    full_message = f"[{timestamp}] {message}"
    
    if to_console:
        print(full_message)
    
    if to_file:
        with open(log_file, 'a', encoding='utf-8') as f:
            f.write(full_message + '\n')

def get_gz_files():
    """获取所有edge_part_XXX.gz文件并按编号排序"""
    pattern = re.compile(r'edge_part_(\d+)\.gz$')
    files = []
    
    for filename in os.listdir(SOURCE_FOLDER):
        match = pattern.match(filename)
        if match:
            file_number = int(match.group(1))
            filepath = os.path.join(SOURCE_FOLDER, filename)
            files.append((file_number, filename, filepath))
    
    # 按文件编号排序
    files.sort(key=lambda x: x[0])
    
    log_message(f"找到 {len(files)} 个edge_part_XXX.gz文件")
    if len(files) > 0:
        log_message(f"文件范围: {files[0][0]} - {files[-1][0]}")
    
    return files

def load_gz_file(filepath, filename=""):
    """安全加载gz压缩的pickle文件"""
    try:
        file_size = os.path.getsize(filepath)
        if file_size == 0:
            return None, f"空文件: {file_size}字节"
        
        # 尝试不同的加载方法
        try_list = [
            # 方法1：正常加载
            lambda: (pickle.load(gzip.open(filepath, 'rb')), "正常加载"),
            
            # 方法2：指定协议版本
            lambda: (pickle.load(gzip.open(filepath, 'rb'), protocol=PICKLE_PROTOCOL), f"指定协议{PICKLE_PROTOCOL}"),
            
            # 方法3：处理可能的编码问题
            lambda: (pickle.load(gzip.open(filepath, 'rb'), encoding='latin-1'), "latin-1编码"),
            
            # 方法4：处理Python 2.x创建的pickle
            lambda: (pickle.load(gzip.open(filepath, 'rb'), encoding='bytes'), "bytes编码"),
            
            # 方法5：先解压到内存再加载
            lambda: (
                pickle.loads(gzip.open(filepath, 'rb').read()), 
                "先解压后加载"
            ),
        ]
        
        for i, load_func in enumerate(try_list):
            try:
                data, method = load_func()
                
                # 验证数据格式
                if data is not None:
                    if isinstance(data, (list, dict, tuple, set)):
                        log_message(f"成功加载 {filename}: 使用{method}, 数据类型: {type(data).__name__}", to_console=False)
                        return data, ""
                    else:
                        # 尝试转换为列表
                        try:
                            data_list = list(data)
                            log_message(f"成功加载 {filename}: 使用{method}, 转换为列表, 长度: {len(data_list)}", to_console=False)
                            return data_list, ""
                        except:
                            return None, f"无法识别的数据类型: {type(data).__name__}"
            except Exception as e:
                if i == len(try_list) - 1:  # 最后一种方法也失败
                    return None, f"所有加载方法都失败: {str(e)}"
                continue
        
        return None, "未知错误"
        
    except Exception as e:
        return None, f"加载失败: {str(e)}"

def merge_gz_files_streaming():
    """流式合并edge_part_XXX.gz文件"""
    
    # 获取所有gz文件
    log_message("开始扫描edge_part_XXX.gz文件...")
    all_files = get_gz_files()
    
    total_files = len(all_files)
    log_message(f"找到 {total_files} 个gz文件")
    
    if total_files == 0:
        log_message("没有找到edge_part_XXX.gz文件")
        return 0, 0, 0
    
    start_time = time.time()
    total_edges = 0
    empty_files = 0
    error_files = 0
    processed_files = 0
    
    # 创建临时目录用于分批处理
    temp_dir = os.path.join(SOURCE_FOLDER, "temp_merge_" + datetime.now().strftime("%Y%m%d_%H%M%S"))
    if not os.path.exists(temp_dir):
        os.makedirs(temp_dir)
    
    try:
        # ========== 第一阶段：直接处理并写入临时文件 ==========
        log_message(f"开始处理文件，批次大小: {BATCH_SIZE}...")
        
        total_batches = (total_files + BATCH_SIZE - 1) // BATCH_SIZE
        current_batch_data = []
        current_batch_edges = 0
        batch_idx = 0
        temp_chunks = []
        
        # 显示文件编号范围
        if total_files > 0:
            log_message(f"文件编号范围: {all_files[0][0]} - {all_files[-1][0]}")
        
        for idx, (file_number, filename, filepath) in enumerate(all_files):
            try:
                # 加载gz文件
                data, error_msg = load_gz_file(filepath, filename)
                
                if data is None:
                    empty_files += 1
                    log_message(f"空/错误文件 {filename}: {error_msg}", to_console=False)
                    continue
                
                # 处理数据，确保是边列表格式
                edges = []
                
                if isinstance(data, list):
                    edges = data
                elif isinstance(data, dict):
                    # 将字典转换为边列表
                    for key, value in data.items():
                        if isinstance(value, list):
                            for v in value:
                                edges.append((key, v))
                        else:
                            edges.append((key, value))
                elif hasattr(data, '__iter__'):
                    # 其他可迭代对象
                    edges = list(data)
                else:
                    # 单个数据项
                    edges = [data]
                
                edge_count = len(edges)
                
                if edge_count == 0:
                    empty_files += 1
                    log_message(f"空数据文件 {filename}", to_console=False)
                    continue
                
                # 添加到当前批次
                current_batch_data.extend(edges)
                current_batch_edges += edge_count
                total_edges += edge_count
                
                # 检查是否需要保存当前批次到临时文件
                if (len(current_batch_data) >= MAX_EDGES_PER_CHUNK or 
                    (idx + 1) % BATCH_SIZE == 0 or 
                    idx == total_files - 1):
                    
                    if current_batch_data:
                        # 保存临时文件
                        temp_file = os.path.join(temp_dir, f"chunk_{batch_idx:05d}.pkl")
                        
                        # 使用统一的协议保存
                        with open(temp_file, 'wb') as f:
                            pickle.dump(current_batch_data, f, protocol=PICKLE_PROTOCOL)
                        
                        temp_chunks.append(temp_file)
                        batch_idx += 1
                        
                        log_message(f"保存临时文件 chunk_{batch_idx:05d}.pkl: {len(current_batch_data):,d} 条边")
                        
                        # 清理内存
                        current_batch_data = []
                        gc.collect()
                
                processed_files += 1
                
                # 显示进度（每50个文件显示一次）
                if (idx + 1) % 50 == 0 or idx == total_files - 1:
                    elapsed = time.time() - start_time
                    progress = (idx + 1) / total_files * 100
                    
                    # 估算剩余时间
                    if idx > 0:
                        time_per_file = elapsed / (idx + 1)
                        remaining_files = total_files - (idx + 1)
                        remaining_time = time_per_file * remaining_files
                        time_str = f"剩余: {remaining_time/60:.1f}分钟"
                    else:
                        time_str = ""
                    
                    log_message(f"进度: {idx + 1}/{total_files} ({progress:.1f}%), "
                               f"边数: {total_edges:,d}, 用时: {elapsed:.1f}秒 {time_str}")
                    
            except Exception as e:
                error_files += 1
                log_message(f"处理文件 {filename} 时出错: {str(e)[:200]}", to_console=False)
                processed_files += 1
        
        # ========== 第二阶段：合并临时文件到最终gzip文件 ==========
        log_message(f"\n开始合并 {len(temp_chunks)} 个临时文件到最终文件...")
        
        if len(temp_chunks) == 0:
            log_message("没有有效数据可以合并")
            # 清理临时目录
            try:
                os.rmdir(temp_dir)
            except:
                pass
            return 0, empty_files, error_files
        
        merge_start_time = time.time()
        
        # 写入最终文件
        with gzip.open(OUTPUT_FILE, 'wb', compresslevel=1) as out_f:
            # 创建一个空的列表来收集所有数据
            final_data = []
            
            for chunk_idx, chunk_file in enumerate(temp_chunks):
                try:
                    # 读取临时文件
                    with open(chunk_file, 'rb') as f:
                        chunk_data = pickle.load(f)
                    
                    # 添加到最终数据
                    final_data.extend(chunk_data)
                    
                    # 如果数据量太大，分批写入
                    if len(final_data) >= 1000000:  # 每100万条边写入一次
                        pickle.dump(final_data, out_f, protocol=PICKLE_PROTOCOL)
                        final_data = []
                        gc.collect()
                    
                    # 显示进度
                    if (chunk_idx + 1) % 10 == 0:
                        progress = (chunk_idx + 1) / len(temp_chunks) * 100
                        log_message(f"合并进度: {chunk_idx + 1}/{len(temp_chunks)} ({progress:.1f}%)")
                    
                    # 删除临时文件
                    try:
                        os.remove(chunk_file)
                    except:
                        pass
                    
                except Exception as e:
                    log_message(f"处理临时文件 {chunk_file} 时出错: {e}")
            
            # 写入剩余数据
            if final_data:
                pickle.dump(final_data, out_f, protocol=PICKLE_PROTOCOL)
        
        # 清理临时目录
        try:
            os.rmdir(temp_dir)
        except:
            # 如果目录不为空，删除所有剩余文件
            for file in os.listdir(temp_dir):
                try:
                    os.remove(os.path.join(temp_dir, file))
                except:
                    pass
            try:
                os.rmdir(temp_dir)
            except:
                pass
        
        merge_time = time.time() - merge_start_time
        total_time = time.time() - start_time
        
        # ========== 验证最终文件 ==========
        log_message("\n验证最终文件...")
        try:
            with gzip.open(OUTPUT_FILE, 'rb') as f:
                # 尝试读取数据
                final_data_check = pickle.load(f)
                final_edge_count = len(final_data_check)
                
                # 验证数据一致性
                if final_edge_count == total_edges:
                    log_message(f"✓ 验证通过: 最终文件包含 {final_edge_count:,d} 条边")
                else:
                    log_message(f"⚠ 警告: 最终文件边数({final_edge_count:,d})与统计数({total_edges:,d})不一致")
                
                # 检查数据格式
                if final_edge_count > 0:
                    sample = final_data_check[0]
                    log_message(f"数据格式: {type(sample).__name__}")
                    
                    # 如果是元组或列表，显示结构
                    if isinstance(sample, (tuple, list)) and len(sample) > 0:
                        log_message(f"边结构: {len(sample)}个元素, 样例: {sample}")
                
                del final_data_check
                
        except Exception as e:
            log_message(f"✗ 最终文件验证失败: {e}")
            # 尝试诊断问题
            try:
                file_size = os.path.getsize(OUTPUT_FILE)
                log_message(f"文件大小: {file_size} 字节 ({file_size/1024/1024:.2f} MB)")
            except:
                pass
        
        # ========== 生成报告 ==========
        log_message("\n" + "="*60)
        log_message("合并完成报告")
        log_message("="*60)
        log_message(f"源文件总数: {total_files}")
        log_message(f"成功处理: {processed_files - empty_files - error_files}")
        log_message(f"空文件数: {empty_files}")
        log_message(f"错误文件数: {error_files}")
        log_message(f"总边数: {total_edges:,d}")
        log_message(f"输出文件: {OUTPUT_FILE}")
        
        try:
            output_size = os.path.getsize(OUTPUT_FILE)
            log_message(f"输出文件大小: {output_size:,d} 字节 ({output_size/1024/1024:.2f} MB)")
            if total_edges > 0:
                avg_edge_size = output_size / total_edges
                log_message(f"平均每条边: {avg_edge_size:.2f} 字节")
        except:
            pass
        
        log_message(f"处理耗时: {total_time:.1f} 秒 ({total_time/60:.1f} 分钟)")
        log_message(f"处理速度: {total_files/total_time:.1f} 文件/秒")
        if total_edges > 0:
            log_message(f"处理速度: {total_edges/total_time:.0f} 边/秒")
        log_message(f"临时文件数: {len(temp_chunks)}")
        log_message(f"合并耗时: {merge_time:.1f} 秒")
        log_message("="*60)
        
        return total_edges, empty_files, error_files
        
    except Exception as e:
        log_message(f"合并过程中发生严重错误: {e}")
        import traceback
        traceback.print_exc()
        
        # 清理临时目录
        try:
            for file in os.listdir(temp_dir):
                os.remove(os.path.join(temp_dir, file))
            os.rmdir(temp_dir)
        except:
            pass
        
        return 0, empty_files, error_files

def merge_gz_files_direct():
    """直接合并方法，适用于数据量不大的情况"""
    log_message("使用直接合并方法...")
    
    # 获取所有gz文件
    all_files = get_gz_files()
    total_files = len(all_files)
    
    if total_files == 0:
        log_message("没有找到edge_part_XXX.gz文件")
        return 0, 0, 0
    
    start_time = time.time()
    all_edges = []
    empty_count = 0
    error_count = 0
    
    log_message(f"开始处理 {total_files} 个文件...")
    
    for idx, (file_number, filename, filepath) in enumerate(all_files):
        try:
            data, error_msg = load_gz_file(filepath, filename)
            
            if data is None:
                empty_count += 1
                continue
            
            # 转换为边列表
            edges = []
            if isinstance(data, list):
                edges = data
            elif isinstance(data, dict):
                for key, value in data.items():
                    if isinstance(value, list):
                        for v in value:
                            edges.append((key, v))
                    else:
                        edges.append((key, value))
            elif hasattr(data, '__iter__'):
                edges = list(data)
            else:
                edges = [data]
            
            if edges:
                all_edges.extend(edges)
            else:
                empty_count += 1
                
        except Exception as e:
            error_count += 1
            log_message(f"处理文件 {filename} 时出错: {e}", to_console=False)
        
        # 显示进度
        if (idx + 1) % 50 == 0 or idx == total_files - 1:
            elapsed = time.time() - start_time
            progress = (idx + 1) / total_files * 100
            log_message(f"进度: {idx + 1}/{total_files} ({progress:.1f}%), "
                       f"边数: {len(all_edges):,d}, 用时: {elapsed:.1f}秒")
    
    # 写入最终文件
    log_message(f"\n写入最终文件，共 {len(all_edges):,d} 条边...")
    
    try:
        with gzip.open(OUTPUT_FILE, 'wb', compresslevel=1) as f:
            pickle.dump(all_edges, f, protocol=PICKLE_PROTOCOL)
        
        log_message(f"成功写入 {OUTPUT_FILE}")
        
        # 验证
        with gzip.open(OUTPUT_FILE, 'rb') as f:
            final_data = pickle.load(f)
            log_message(f"验证通过: {len(final_data):,d} 条边")
            del final_data
            
    except Exception as e:
        log_message(f"写入失败: {e}")
        return 0, empty_count, error_count
    
    total_time = time.time() - start_time
    log_message(f"完成! 耗时: {total_time:.1f}秒")
    
    return len(all_edges), empty_count, error_count

# ========== 主程序入口 ==========
if __name__ == "__main__":
    # 显示程序信息
    log_message("="*60)
    log_message("EDGE_PART_GZ文件合并工具")
    log_message(f"Python版本: {sys.version}")
    log_message(f"Pickle最高协议: {PICKLE_PROTOCOL}")
    log_message(f"源目录: {SOURCE_FOLDER}")
    log_message(f"输出文件: {OUTPUT_FILE}")
    log_message("="*60)
    
    # 检查目录
    if not os.path.exists(SOURCE_FOLDER):
        log_message(f"错误: 源目录不存在 - {SOURCE_FOLDER}")
        sys.exit(1)
    
    # 确保输出目录存在
    output_dir = os.path.dirname(OUTPUT_FILE)
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    # 选择合并方法
    log_message("\n请选择合并方法:")
    log_message("1. 流式分批处理 (推荐，适合大量文件)")
    log_message("2. 直接合并 (适合数据量不大的情况)")
    
    try:
        choice = input("请输入选择 (1 或 2，默认1): ").strip()
        if choice == "":
            choice = "1"
    except:
        choice = "1"
    
    if choice == "2":
        log_message("使用直接合并方法...")
        total_edges, empty_files, error_files = merge_gz_files_direct()
    else:
        log_message("使用流式分批处理...")
        total_edges, empty_files, error_files = merge_gz_files_streaming()
    
    # 最终状态
    log_message("\n" + "="*60)
    if total_edges > 0:
        log_message(f"✓ 合并成功! 共合并 {total_edges:,d} 条边")
        log_message(f"✓ 输出文件: {OUTPUT_FILE}")
        
        # 检查文件大小
        try:
            file_size = os.path.getsize(OUTPUT_FILE)
            log_message(f"✓ 文件大小: {file_size/1024/1024:.2f} MB")
        except:
            pass
    else:
        log_message("⚠ 合并完成，但未生成有效数据")
    
    log_message("="*60)
    
    # 等待用户确认
    if sys.platform == "win32":
        try:
            input("\n按Enter键退出...")
        except:
            pass

In [ ]:
import gzip
import os
import time
from datetime import datetime
import pickle
import gc
import sys
import re
import json

# ========== 配置区域 ==========
SOURCE_FOLDER = r"H:\ASIST\entities_pair"
OUTPUT_FILE = r"G:\output\merged_edgess.pkl.gz" 
LOG_FOLDER = r"H:\ASIST\logs"
CHECKPOINT_FOLDER = r"H:\ASIST\checkpoints"  # 断点记录文件夹

# 处理参数
BATCH_SIZE = 100  # 每批处理的文件数量（减少因为gz文件可能更大）
MAX_EDGES_PER_CHUNK = 50000  # 每个临时文件的最大边数
PICKLE_PROTOCOL = pickle.HIGHEST_PROTOCOL  # 统一使用最高协议
CHECKPOINT_INTERVAL = 50  # 每处理多少个文件保存一次断点
MAX_BATCH_FILES_PER_TEMP = 100  # 每个temp文件夹最多处理100个batch文件

# ========== 初始化 ==========
if not os.path.exists(LOG_FOLDER):
    os.makedirs(LOG_FOLDER)

if not os.path.exists(CHECKPOINT_FOLDER):
    os.makedirs(CHECKPOINT_FOLDER)

log_file = os.path.join(LOG_FOLDER, f'log_merge_concept_pairs_{datetime.now().strftime("%Y%m%d_%H%M%S")}.txt')

def log_message(message, to_console=True, to_file=True):
    """记录日志"""
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    full_message = f"[{timestamp}] {message}"
    
    if to_console:
        print(full_message)
    
    if to_file:
        with open(log_file, 'a', encoding='utf-8') as f:
            f.write(full_message + '\n')

def get_gz_files():
    """获取所有temp_XXX文件夹中的batch_XXXXX.pkl.gz文件并按编号排序，每个temp文件夹最多取前MAX_BATCH_FILES_PER_TEMP个文件"""
    pattern = re.compile(r'temp_(\d+)$')  # 匹配temp_XXX文件夹
    batch_pattern = re.compile(r'batch_(\d+)\.pkl\.gz$')  # 匹配batch_XXXXX.pkl.gz文件
    
    all_batch_files = []
    temp_folders_info = []
    
    # 遍历SOURCE_FOLDER下的所有temp_XXX文件夹
    for item in os.listdir(SOURCE_FOLDER):
        folder_path = os.path.join(SOURCE_FOLDER, item)
        match = pattern.match(item)
        
        if match and os.path.isdir(folder_path):
            temp_number = int(match.group(1))
            
            # 获取该文件夹中的所有batch文件
            batch_files_in_temp = []
            for filename in os.listdir(folder_path):
                batch_match = batch_pattern.match(filename)
                if batch_match:
                    batch_number = int(batch_match.group(1))
                    filepath = os.path.join(folder_path, filename)
                    batch_files_in_temp.append({
                        'temp_num': temp_number,
                        'batch_num': batch_number,
                        'filename': filename,
                        'filepath': filepath,
                        'sort_key': batch_number
                    })
            
            # 按batch编号排序
            batch_files_in_temp.sort(key=lambda x: x['sort_key'])
            
            # 只取前MAX_BATCH_FILES_PER_TEMP个文件
            original_count = len(batch_files_in_temp)
            if original_count > MAX_BATCH_FILES_PER_TEMP:
                batch_files_in_temp = batch_files_in_temp[:MAX_BATCH_FILES_PER_TEMP]
                log_message(f"temp_{temp_number}: 共有 {original_count} 个batch文件，只取前 {MAX_BATCH_FILES_PER_TEMP} 个 (batch_0 到 batch_{MAX_BATCH_FILES_PER_TEMP-1})")
            
            # 添加到总列表
            for batch_file in batch_files_in_temp:
                all_batch_files.append(batch_file)
            
            temp_folders_info.append({
                'temp_num': temp_number,
                'total_files': original_count,
                'selected_files': len(batch_files_in_temp)
            })
    
    # 按temp编号和batch编号排序
    all_batch_files.sort(key=lambda x: (x['temp_num'], x['batch_num']))
    
    # 显示每个temp文件夹的处理情况
    log_message(f"\n各temp文件夹文件统计:")
    for info in temp_folders_info:
        log_message(f"  temp_{info['temp_num']}: {info['selected_files']}/{info['total_files']} 个文件被选中")
    
    log_message(f"\n总共选中 {len(all_batch_files)} 个batch_XXXXX.pkl.gz文件")
    if len(all_batch_files) > 0:
        log_message(f"temp文件夹范围: {all_batch_files[0]['temp_num']} - {all_batch_files[-1]['temp_num']}")
        log_message(f"batch文件范围: {all_batch_files[0]['batch_num']} - {all_batch_files[-1]['batch_num']}")
    
    return all_batch_files

def save_checkpoint(checkpoint_file, processed_files, total_edges, current_batch_data, 
                   batch_idx, temp_chunks, completed_files_list):
    """保存断点信息"""
    try:
        # 创建临时文件，避免写入过程中断导致文件损坏
        temp_checkpoint = checkpoint_file + '.tmp'
        
        checkpoint_data = {
            'processed_files': processed_files,
            'total_edges': total_edges,
            'batch_idx': batch_idx,
            'temp_chunks': temp_chunks,
            'completed_files': completed_files_list,
            'timestamp': datetime.now().isoformat(),
            'output_file': OUTPUT_FILE
        }
        
        # 保存当前批次数据（可能需要单独保存，因为可能很大）
        if current_batch_data and len(current_batch_data) > 0:
            batch_data_file = checkpoint_file.replace('.json', '_batch_data.pkl')
            with open(batch_data_file, 'wb') as f:
                pickle.dump(current_batch_data, f, protocol=PICKLE_PROTOCOL)
            checkpoint_data['batch_data_file'] = batch_data_file
        else:
            checkpoint_data['batch_data_file'] = None
        
        with open(temp_checkpoint, 'w', encoding='utf-8') as f:
            json.dump(checkpoint_data, f, indent=2, ensure_ascii=False)
        
        # 原子操作：重命名临时文件
        os.replace(temp_checkpoint, checkpoint_file)
        
        log_message(f"断点已保存: 已处理 {processed_files} 个文件, {total_edges:,d} 条边", to_console=False)
        return True
    except Exception as e:
        log_message(f"保存断点失败: {e}", to_console=False)
        return False

def load_checkpoint(checkpoint_file):
    """加载断点信息"""
    if not os.path.exists(checkpoint_file):
        return None
    
    try:
        with open(checkpoint_file, 'r', encoding='utf-8') as f:
            checkpoint_data = json.load(f)
        
        # 加载批次数据
        if checkpoint_data.get('batch_data_file') and os.path.exists(checkpoint_data['batch_data_file']):
            with open(checkpoint_data['batch_data_file'], 'rb') as f:
                checkpoint_data['current_batch_data'] = pickle.load(f)
        else:
            checkpoint_data['current_batch_data'] = []
        
        log_message(f"加载断点: 已处理 {checkpoint_data['processed_files']} 个文件, "
                   f"{checkpoint_data['total_edges']:,d} 条边")
        return checkpoint_data
    except Exception as e:
        log_message(f"加载断点失败: {e}")
        return None

def load_gz_file(filepath, filename=""):
    """安全加载gz压缩的pickle文件"""
    try:
        file_size = os.path.getsize(filepath)
        if file_size == 0:
            return None, f"空文件: {file_size}字节"
        
        # 尝试不同的加载方法
        try_list = [
            # 方法1：正常加载
            lambda: (pickle.load(gzip.open(filepath, 'rb')), "正常加载"),
            
            # 方法2：指定协议版本
            lambda: (pickle.load(gzip.open(filepath, 'rb'), protocol=PICKLE_PROTOCOL), f"指定协议{PICKLE_PROTOCOL}"),
            
            # 方法3：处理可能的编码问题
            lambda: (pickle.load(gzip.open(filepath, 'rb'), encoding='latin-1'), "latin-1编码"),
            
            # 方法4：处理Python 2.x创建的pickle
            lambda: (pickle.load(gzip.open(filepath, 'rb'), encoding='bytes'), "bytes编码"),
            
            # 方法5：先解压到内存再加载
            lambda: (
                pickle.loads(gzip.open(filepath, 'rb').read()), 
                "先解压后加载"
            ),
        ]
        
        for i, load_func in enumerate(try_list):
            try:
                data, method = load_func()
                
                # 验证数据格式
                if data is not None:
                    if isinstance(data, (list, dict, tuple, set)):
                        log_message(f"成功加载 {filename}: 使用{method}, 数据类型: {type(data).__name__}", to_console=False)
                        return data, ""
                    else:
                        # 尝试转换为列表
                        try:
                            data_list = list(data)
                            log_message(f"成功加载 {filename}: 使用{method}, 转换为列表, 长度: {len(data_list)}", to_console=False)
                            return data_list, ""
                        except:
                            return None, f"无法识别的数据类型: {type(data).__name__}"
            except Exception as e:
                if i == len(try_list) - 1:  # 最后一种方法也失败
                    return None, f"所有加载方法都失败: {str(e)}"
                continue
        
        return None, "未知错误"
        
    except Exception as e:
        return None, f"加载失败: {str(e)}"

def merge_gz_files_streaming(resume=True):
    """流式合并temp_XXX文件夹中的batch_XXXXX.pkl.gz文件，支持断点续传，每个temp文件夹最多处理MAX_BATCH_FILES_PER_TEMP个文件"""
    
    # 获取所有gz文件
    log_message("开始扫描temp_XXX文件夹中的batch_XXXXX.pkl.gz文件...")
    all_files = get_gz_files()
    
    total_files = len(all_files)
    log_message(f"找到 {total_files} 个batch文件（每个temp文件夹最多{MAX_BATCH_FILES_PER_TEMP}个）")
    
    if total_files == 0:
        log_message("没有找到batch_XXXXX.pkl.gz文件")
        return 0, 0, 0
    
    # 生成唯一的合并任务ID（基于输出文件路径）
    task_id = re.sub(r'[^\w\-_]', '_', OUTPUT_FILE)
    checkpoint_file = os.path.join(CHECKPOINT_FOLDER, f'checkpoint_{task_id}.json')
    
    # 加载断点
    processed_files_count = 0
    total_edges = 0
    current_batch_data = []
    batch_idx = 0
    temp_chunks = []
    completed_files_set = set()
    
    if resume:
        checkpoint = load_checkpoint(checkpoint_file)
        if checkpoint:
            processed_files_count = checkpoint['processed_files']
            total_edges = checkpoint['total_edges']
            batch_idx = checkpoint['batch_idx']
            temp_chunks = checkpoint['temp_chunks']
            completed_files_set = set(checkpoint['completed_files'])
            current_batch_data = checkpoint.get('current_batch_data', [])
            
            log_message(f"从断点恢复: 已处理 {processed_files_count}/{total_files} 个文件")
            
            # 验证临时文件是否存在
            existing_chunks = []
            for chunk in temp_chunks:
                if os.path.exists(chunk):
                    existing_chunks.append(chunk)
                else:
                    log_message(f"警告: 临时文件不存在 {chunk}", to_console=False)
            temp_chunks = existing_chunks
            batch_idx = len(temp_chunks)
    
    start_time = time.time()
    empty_files = 0
    error_files = 0
    processed_files = 0
    
    # 创建临时目录用于分批处理
    temp_dir = os.path.join(SOURCE_FOLDER, "temp_merge_" + datetime.now().strftime("%Y%m%d_%H%M%S"))
    if not os.path.exists(temp_dir):
        os.makedirs(temp_dir)
    
    # 如果有已存在的临时文件，需要移动到新的临时目录
    if temp_chunks:
        log_message(f"移动 {len(temp_chunks)} 个已有临时文件到新临时目录...")
        new_temp_chunks = []
        for old_chunk in temp_chunks:
            if os.path.exists(old_chunk):
                new_chunk = os.path.join(temp_dir, os.path.basename(old_chunk))
                os.rename(old_chunk, new_chunk)
                new_temp_chunks.append(new_chunk)
        temp_chunks = new_temp_chunks
    
    try:
        # ========== 第一阶段：直接处理并写入临时文件 ==========
        log_message(f"开始处理文件，批次大小: {BATCH_SIZE}...")
        
        total_batches = (total_files + BATCH_SIZE - 1) // BATCH_SIZE
        
        # 显示文件编号范围
        if total_files > 0:
            log_message(f"temp文件夹范围: {all_files[0]['temp_num']} - {all_files[-1]['temp_num']}")
            log_message(f"batch文件范围: {all_files[0]['batch_num']} - {all_files[-1]['batch_num']}")
        
        for idx, file_info in enumerate(all_files):
            # 跳过已处理的文件
            if idx < processed_files_count:
                continue
            
            # 检查文件是否已完成（基于文件路径的唯一标识）
            file_key = f"{file_info['temp_num']}_{file_info['batch_num']}"
            if file_key in completed_files_set:
                log_message(f"跳过已处理文件: {file_info['filename']}", to_console=False)
                processed_files += 1
                continue
            
            try:
                filename = file_info['filename']
                filepath = file_info['filepath']
                temp_num = file_info['temp_num']
                batch_num = file_info['batch_num']
                
                # 加载gz文件
                data, error_msg = load_gz_file(filepath, filename)
                
                if data is None:
                    empty_files += 1
                    log_message(f"空/错误文件 temp_{temp_num}/{filename}: {error_msg}", to_console=False)
                    completed_files_set.add(file_key)
                    processed_files += 1
                    continue
                
                # 处理数据，确保是边列表格式
                edges = []
                
                if isinstance(data, list):
                    edges = data
                elif isinstance(data, dict):
                    # 将字典转换为边列表
                    for key, value in data.items():
                        if isinstance(value, list):
                            for v in value:
                                edges.append((key, v))
                        else:
                            edges.append((key, value))
                elif hasattr(data, '__iter__'):
                    # 其他可迭代对象
                    edges = list(data)
                else:
                    # 单个数据项
                    edges = [data]
                
                edge_count = len(edges)
                
                if edge_count == 0:
                    empty_files += 1
                    log_message(f"空数据文件 temp_{temp_num}/{filename}", to_console=False)
                    completed_files_set.add(file_key)
                    processed_files += 1
                    continue
                
                # 添加到当前批次
                current_batch_data.extend(edges)
                current_batch_edges = edge_count
                total_edges += edge_count
                
                # 检查是否需要保存当前批次到临时文件
                if (len(current_batch_data) >= MAX_EDGES_PER_CHUNK or 
                    (idx + 1) % BATCH_SIZE == 0 or 
                    idx == total_files - 1):
                    
                    if current_batch_data:
                        # 保存临时文件
                        temp_file = os.path.join(temp_dir, f"chunk_{batch_idx:05d}.pkl")
                        
                        # 使用统一的协议保存
                        with open(temp_file, 'wb') as f:
                            pickle.dump(current_batch_data, f, protocol=PICKLE_PROTOCOL)
                        
                        temp_chunks.append(temp_file)
                        batch_idx += 1
                        
                        log_message(f"保存临时文件 chunk_{batch_idx:05d}.pkl: {len(current_batch_data):,d} 条边")
                        
                        # 清理内存
                        current_batch_data = []
                        gc.collect()
                
                # 记录已完成的文件
                completed_files_set.add(file_key)
                processed_files += 1
                
                # 保存断点（每处理CHECKPOINT_INTERVAL个文件保存一次）
                if (processed_files - processed_files_count) % CHECKPOINT_INTERVAL == 0 or processed_files == total_files:
                    save_checkpoint(checkpoint_file, processed_files, total_edges, 
                                  current_batch_data, batch_idx, temp_chunks, 
                                  list(completed_files_set))
                
                # 显示进度（每50个文件显示一次）
                if (idx + 1) % 50 == 0 or idx == total_files - 1:
                    elapsed = time.time() - start_time
                    progress = (idx + 1) / total_files * 100
                    
                    # 估算剩余时间
                    if idx > processed_files_count:
                        time_per_file = elapsed / (idx + 1 - processed_files_count)
                        remaining_files = total_files - (idx + 1)
                        remaining_time = time_per_file * remaining_files
                        time_str = f"剩余: {remaining_time/60:.1f}分钟"
                    else:
                        time_str = ""
                    
                    log_message(f"进度: {idx + 1}/{total_files} ({progress:.1f}%), "
                               f"边数: {total_edges:,d}, 用时: {elapsed:.1f}秒 {time_str}")
                    
            except Exception as e:
                error_files += 1
                log_message(f"处理文件 {file_info['filename']} 时出错: {str(e)[:200]}", to_console=False)
                processed_files += 1
        
        # ========== 第二阶段：合并临时文件到最终gzip文件 ==========
        log_message(f"\n开始合并 {len(temp_chunks)} 个临时文件到最终文件...")
        
        if len(temp_chunks) == 0:
            log_message("没有有效数据可以合并")
            # 清理临时目录
            try:
                os.rmdir(temp_dir)
            except:
                pass
            return 0, empty_files, error_files
        
        merge_start_time = time.time()
        
        # 检查是否已有部分合并的输出文件
        output_mode = 'ab' if resume and os.path.exists(OUTPUT_FILE) else 'wb'
        
        # 写入最终文件
        with gzip.open(OUTPUT_FILE, output_mode, compresslevel=1) as out_f:
            # 如果是追加模式，需要先读取已写入的数据结构
            if output_mode == 'ab':
                log_message("追加模式：保留已有数据")
            else:
                log_message("新建模式：创建新文件")
            
            # 创建一个空的列表来收集所有数据
            final_data = []
            processed_chunks = set()
            
            # 如果是续传，记录已处理的临时文件
            if resume and os.path.exists(checkpoint_file):
                try:
                    with open(checkpoint_file, 'r', encoding='utf-8') as f:
                        merge_checkpoint = json.load(f)
                        processed_chunks = set(merge_checkpoint.get('processed_chunks', []))
                except:
                    pass
            
            for chunk_idx, chunk_file in enumerate(temp_chunks):
                # 跳过已处理的chunk
                if chunk_file in processed_chunks:
                    log_message(f"跳过已处理临时文件: {os.path.basename(chunk_file)}", to_console=False)
                    continue
                    
                try:
                    # 读取临时文件
                    with open(chunk_file, 'rb') as f:
                        chunk_data = pickle.load(f)
                    
                    # 添加到最终数据
                    final_data.extend(chunk_data)
                    
                    # 如果数据量太大，分批写入
                    if len(final_data) >= 1000000:  # 每100万条边写入一次
                        pickle.dump(final_data, out_f, protocol=PICKLE_PROTOCOL)
                        final_data = []
                        gc.collect()
                    
                    # 记录已处理的chunk
                    processed_chunks.add(chunk_file)
                    
                    # 保存合并进度断点
                    merge_checkpoint = {
                        'processed_chunks': list(processed_chunks),
                        'total_chunks': len(temp_chunks),
                        'timestamp': datetime.now().isoformat()
                    }
                    with open(checkpoint_file + '_merge.json', 'w', encoding='utf-8') as f:
                        json.dump(merge_checkpoint, f, indent=2)
                    
                    # 显示进度
                    if (chunk_idx + 1) % 10 == 0 or chunk_idx == len(temp_chunks) - 1:
                        progress = (chunk_idx + 1) / len(temp_chunks) * 100
                        log_message(f"合并进度: {chunk_idx + 1}/{len(temp_chunks)} ({progress:.1f}%)")
                    
                    # 删除临时文件
                    try:
                        os.remove(chunk_file)
                    except:
                        pass
                    
                except Exception as e:
                    log_message(f"处理临时文件 {chunk_file} 时出错: {e}")
            
            # 写入剩余数据
            if final_data:
                pickle.dump(final_data, out_f, protocol=PICKLE_PROTOCOL)
        
        # 清理临时目录
        try:
            os.rmdir(temp_dir)
        except:
            # 如果目录不为空，删除所有剩余文件
            for file in os.listdir(temp_dir):
                try:
                    os.remove(os.path.join(temp_dir, file))
                except:
                    pass
            try:
                os.rmdir(temp_dir)
            except:
                pass
        
        # 删除断点文件（合并完成）
        try:
            os.remove(checkpoint_file)
            merge_checkpoint_file = checkpoint_file + '_merge.json'
            if os.path.exists(merge_checkpoint_file):
                os.remove(merge_checkpoint_file)
        except:
            pass
        
        merge_time = time.time() - merge_start_time
        total_time = time.time() - start_time
        
        # ========== 验证最终文件 ==========
        log_message("\n验证最终文件...")
        try:
            with gzip.open(OUTPUT_FILE, 'rb') as f:
                # 尝试读取数据
                final_data_check = pickle.load(f)
                final_edge_count = len(final_data_check)
                
                # 验证数据一致性
                if final_edge_count == total_edges:
                    log_message(f"✓ 验证通过: 最终文件包含 {final_edge_count:,d} 条边")
                else:
                    log_message(f"⚠ 警告: 最终文件边数({final_edge_count:,d})与统计数({total_edges:,d})不一致")
                
                # 检查数据格式
                if final_edge_count > 0:
                    sample = final_data_check[0]
                    log_message(f"数据格式: {type(sample).__name__}")
                    
                    # 如果是元组或列表，显示结构
                    if isinstance(sample, (tuple, list)) and len(sample) > 0:
                        log_message(f"边结构: {len(sample)}个元素, 样例: {sample}")
                
                del final_data_check
                
        except Exception as e:
            log_message(f"✗ 最终文件验证失败: {e}")
            # 尝试诊断问题
            try:
                file_size = os.path.getsize(OUTPUT_FILE)
                log_message(f"文件大小: {file_size} 字节 ({file_size/1024/1024:.2f} MB)")
            except:
                pass
        
        # ========== 生成报告 ==========
        log_message("\n" + "="*60)
        log_message("合并完成报告")
        log_message("="*60)
        log_message(f"源文件总数: {total_files}")
        log_message(f"成功处理: {processed_files - empty_files - error_files}")
        log_message(f"空文件数: {empty_files}")
        log_message(f"错误文件数: {error_files}")
        log_message(f"总边数: {total_edges:,d}")
        log_message(f"输出文件: {OUTPUT_FILE}")
        log_message(f"每个temp文件夹最多处理文件数: {MAX_BATCH_FILES_PER_TEMP}")
        
        try:
            output_size = os.path.getsize(OUTPUT_FILE)
            log_message(f"输出文件大小: {output_size:,d} 字节 ({output_size/1024/1024:.2f} MB)")
            if total_edges > 0:
                avg_edge_size = output_size / total_edges
                log_message(f"平均每条边: {avg_edge_size:.2f} 字节")
        except:
            pass
        
        log_message(f"处理耗时: {total_time:.1f} 秒 ({total_time/60:.1f} 分钟)")
        log_message(f"处理速度: {total_files/total_time:.1f} 文件/秒")
        if total_edges > 0:
            log_message(f"处理速度: {total_edges/total_time:.0f} 边/秒")
        log_message(f"临时文件数: {len(temp_chunks)}")
        log_message(f"合并耗时: {merge_time:.1f} 秒")
        log_message("="*60)
        
        return total_edges, empty_files, error_files
        
    except Exception as e:
        log_message(f"合并过程中发生严重错误: {e}")
        import traceback
        traceback.print_exc()
        
        # 保存最后的断点
        save_checkpoint(checkpoint_file, processed_files, total_edges, 
                       current_batch_data, batch_idx, temp_chunks, 
                       list(completed_files_set))
        
        # 清理临时目录（可选）
        # 注意：不清空临时文件，以便续传时使用
        
        return 0, empty_files, error_files

def merge_gz_files_direct(resume=True):
    """直接合并方法，支持断点续传，适用于数据量不大的情况，每个temp文件夹最多处理MAX_BATCH_FILES_PER_TEMP个文件"""
    log_message("使用直接合并方法...")
    
    # 获取所有gz文件
    all_files = get_gz_files()
    total_files = len(all_files)
    
    if total_files == 0:
        log_message("没有找到batch_XXXXX.pkl.gz文件")
        return 0, 0, 0
    
    # 生成唯一的合并任务ID（基于输出文件路径）
    task_id = re.sub(r'[^\w\-_]', '_', OUTPUT_FILE)
    checkpoint_file = os.path.join(CHECKPOINT_FOLDER, f'direct_checkpoint_{task_id}.json')
    
    # 加载断点
    processed_files_count = 0
    completed_files_set = set()
    
    if resume:
        if os.path.exists(checkpoint_file):
            try:
                with open(checkpoint_file, 'r', encoding='utf-8') as f:
                    checkpoint = json.load(f)
                    processed_files_count = checkpoint.get('processed_files', 0)
                    completed_files_set = set(checkpoint.get('completed_files', []))
                    log_message(f"从断点恢复: 已处理 {processed_files_count}/{total_files} 个文件")
            except Exception as e:
                log_message(f"加载断点失败: {e}")
    
    start_time = time.time()
    all_edges = []
    empty_count = 0
    error_count = 0
    
    # 如果已有输出文件，尝试加载已有数据
    if resume and os.path.exists(OUTPUT_FILE) and processed_files_count > 0:
        try:
            with gzip.open(OUTPUT_FILE, 'rb') as f:
                all_edges = pickle.load(f)
                log_message(f"加载已有输出文件: {len(all_edges):,d} 条边")
        except Exception as e:
            log_message(f"加载已有输出文件失败: {e}，将重新开始")
            all_edges = []
            processed_files_count = 0
            completed_files_set.clear()
    
    log_message(f"开始处理 {total_files} 个文件...")
    log_message(f"每个temp文件夹最多处理 {MAX_BATCH_FILES_PER_TEMP} 个文件")
    
    for idx, file_info in enumerate(all_files):
        # 跳过已处理的文件
        if idx < processed_files_count:
            continue
        
        file_key = f"{file_info['temp_num']}_{file_info['batch_num']}"
        if file_key in completed_files_set:
            log_message(f"跳过已处理文件: {file_info['filename']}", to_console=False)
            continue
        
        try:
            filename = file_info['filename']
            filepath = file_info['filepath']
            temp_num = file_info['temp_num']
            batch_num = file_info['batch_num']
            
            data, error_msg = load_gz_file(filepath, filename)
            
            if data is None:
                empty_count += 1
                completed_files_set.add(file_key)
                continue
            
            # 转换为边列表
            edges = []
            if isinstance(data, list):
                edges = data
            elif isinstance(data, dict):
                for key, value in data.items():
                    if isinstance(value, list):
                        for v in value:
                            edges.append((key, v))
                    else:
                        edges.append((key, value))
            elif hasattr(data, '__iter__'):
                edges = list(data)
            else:
                edges = [data]
            
            if edges:
                all_edges.extend(edges)
            else:
                empty_count += 1
            
            completed_files_set.add(file_key)
            
            # 保存断点（每处理CHECKPOINT_INTERVAL个文件保存一次）
            if len(completed_files_set) % CHECKPOINT_INTERVAL == 0:
                checkpoint_data = {
                    'processed_files': idx + 1,
                    'completed_files': list(completed_files_set),
                    'timestamp': datetime.now().isoformat()
                }
                with open(checkpoint_file, 'w', encoding='utf-8') as f:
                    json.dump(checkpoint_data, f, indent=2)
                
                # 同时保存中间结果到输出文件
                with gzip.open(OUTPUT_FILE, 'wb', compresslevel=1) as f:
                    pickle.dump(all_edges, f, protocol=PICKLE_PROTOCOL)
                
                log_message(f"断点已保存: 已处理 {idx + 1}/{total_files} 个文件, {len(all_edges):,d} 条边")
                
        except Exception as e:
            error_count += 1
            log_message(f"处理文件 {file_info['filename']} 时出错: {e}", to_console=False)
        
        # 显示进度
        if (idx + 1) % 50 == 0 or idx == total_files - 1:
            elapsed = time.time() - start_time
            progress = (idx + 1) / total_files * 100
            log_message(f"进度: {idx + 1}/{total_files} ({progress:.1f}%), "
                       f"边数: {len(all_edges):,d}, 用时: {elapsed:.1f}秒")
    
    # 写入最终文件
    log_message(f"\n写入最终文件，共 {len(all_edges):,d} 条边...")
    
    try:
        with gzip.open(OUTPUT_FILE, 'wb', compresslevel=1) as f:
            pickle.dump(all_edges, f, protocol=PICKLE_PROTOCOL)
        
        log_message(f"成功写入 {OUTPUT_FILE}")
        
        # 删除断点文件
        try:
            os.remove(checkpoint_file)
        except:
            pass
        
        # 验证
        with gzip.open(OUTPUT_FILE, 'rb') as f:
            final_data = pickle.load(f)
            log_message(f"验证通过: {len(final_data):,d} 条边")
            del final_data
            
    except Exception as e:
        log_message(f"写入失败: {e}")
        return 0, empty_count, error_count
    
    total_time = time.time() - start_time
    log_message(f"完成! 耗时: {total_time:.1f}秒")
    
    return len(all_edges), empty_count, error_count

# ========== 主程序入口 ==========
if __name__ == "__main__":
    # 显示程序信息
    log_message("="*60)
    log_message("BATCH文件合并工具 (支持断点续传)")
    log_message(f"Python版本: {sys.version}")
    log_message(f"Pickle最高协议: {PICKLE_PROTOCOL}")
    log_message(f"源目录: {SOURCE_FOLDER}")
    log_message(f"输出文件: {OUTPUT_FILE}")
    log_message(f"断点目录: {CHECKPOINT_FOLDER}")
    log_message(f"每个temp文件夹最多处理: {MAX_BATCH_FILES_PER_TEMP} 个batch文件")
    log_message("="*60)
    
    # 检查目录
    if not os.path.exists(SOURCE_FOLDER):
        log_message(f"错误: 源目录不存在 - {SOURCE_FOLDER}")
        sys.exit(1)
    
    # 确保输出目录存在
    output_dir = os.path.dirname(OUTPUT_FILE)
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    # 检查是否有未完成的合并任务
    task_id = re.sub(r'[^\w\-_]', '_', OUTPUT_FILE)
    checkpoint_file = os.path.join(CHECKPOINT_FOLDER, f'checkpoint_{task_id}.json')
    has_unfinished = os.path.exists(checkpoint_file)
    
    if has_unfinished:
        log_message("\n检测到未完成的合并任务!")
        log_message(f"断点文件: {checkpoint_file}")
        try:
            with open(checkpoint_file, 'r', encoding='utf-8') as f:
                checkpoint = json.load(f)
                log_message(f"上次进度: 已处理 {checkpoint.get('processed_files', 0)} 个文件, "
                          f"{checkpoint.get('total_edges', 0):,d} 条边")
        except:
            pass
        
        resume_choice = input("是否继续上次的任务? (y/n，默认y): ").strip().lower()
        if resume_choice == '' or resume_choice == 'y' or resume_choice == 'yes':
            resume_flag = True
            log_message("将从中断点继续处理...")
        else:
            resume_flag = False
            log_message("将重新开始处理（忽略之前的进度）...")
    else:
        resume_flag = True
        log_message("未检测到未完成的任务，将开始新的合并...")
    
    # 选择合并方法
    log_message("\n请选择合并方法:")
    log_message("1. 流式分批处理 (推荐，适合大量文件，支持断点续传)")
    log_message("2. 直接合并 (适合数据量不大的情况，支持断点续传)")
    
    try:
        choice = input("请输入选择 (1 或 2，默认1): ").strip()
        if choice == "":
            choice = "1"
    except:
        choice = "1"
    
    if choice == "2":
        log_message("使用直接合并方法...")
        total_edges, empty_files, error_files = merge_gz_files_direct(resume=resume_flag)
    else:
        log_message("使用流式分批处理...")
        total_edges, empty_files, error_files = merge_gz_files_streaming(resume=resume_flag)
    
    # 最终状态
    log_message("\n" + "="*60)
    if total_edges > 0:
        log_message(f"✓ 合并成功! 共合并 {total_edges:,d} 条边")
        log_message(f"✓ 输出文件: {OUTPUT_FILE}")
        
        # 检查文件大小
        try:
            file_size = os.path.getsize(OUTPUT_FILE)
            log_message(f"✓ 文件大小: {file_size/1024/1024:.2f} MB")
        except:
            pass
    else:
        log_message("⚠ 合并完成，但未生成有效数据")
    
    log_message("="*60)
    
    # 等待用户确认
    if sys.platform == "win32":
        try:
            input("\n按Enter键退出...")
        except:
            pass

In [ ]:
import gzip
import pickle

file_path = r"H:\ASIST\entities_pair\temp_816\batch_0069.pkl.gz"

# 读取压缩的pickle文件
with gzip.open(file_path, 'rb') as f:
    data = pickle.load(f)

# 查看数据结构
print(f"数据类型: {type(data)}")
print(f"数据内容: {data}")

In [ ]:
import pandas as pd
import os

# 定义文件路径
input_file = r"H:\ASIST\data_graph\final_dynamic_graph.parquet"
output_file = r"H:\ASIST\data_graph\full_dynamic_graph.csv"

try:
    # 读取parquet文件
    print(f"正在读取文件: {input_file}")
    df = pd.read_parquet(input_file)
    
    # 显示基本信息
    print(f"数据形状: {df.shape}")
    print(f"列名: {list(df.columns)}")
    print(f"数据类型:\n{df.dtypes}")
    
    # 转换为CSV文件
    print(f"正在转换为CSV: {output_file}")
    df.to_csv(output_file, index=False, encoding='utf-8-sig')
    
    print(f"转换完成！文件已保存至: {output_file}")
    
    # 显示文件大小信息
    input_size = os.path.getsize(input_file) / (1024 * 1024)  # 转换为MB
    output_size = os.path.getsize(output_file) / (1024 * 1024)
    print(f"原文件大小: {input_size:.2f} MB")
    print(f"CSV文件大小: {output_size:.2f} MB")
    
except FileNotFoundError:
    print(f"错误：找不到文件 {input_file}")
except Exception as e:
    print(f"转换过程中出现错误: {str(e)}")

In [ ]:
import pandas as pd

# 读取两个parquet文件
df1 = pd.read_parquet(r"G:\data_concept_graph\full_dynamic_graphs.parquet")
df2 = pd.read_parquet(r"G:\data_concept_graph\full_dynamic_graph.parquet")

# 合并两个DataFrame
merged_df = pd.concat([df1, df2], ignore_index=True)

# 保存合并后的DataFrame为parquet文件
merged_df.to_parquet(r'H:\ASIST\data_graph\final_dynamic_graph.parquet', index=False)

print("两个文件已经成功合并")

In [ ]:
import pandas as pd

# 只读取v1和v2列
df = pd.read_parquet(r"E:\study\research\impact4cast_cscl\data_concept_graph\full_dynamic_graph.parquet", columns=['v1', 'v2'])

# 计算不同ID数量
unique_count = len(pd.unique(pd.concat([df['v1'], df['v2']])))

print(f"不同ID总数: {unique_count}")

In [ ]:
import pyarrow.parquet as pq

# 读取parquet文件的元数据
file_path = r"E:\study\research\impact4cast_cscl\data_concept_graph\full_dynamic_graph.parquet"
parquet_file = pq.ParquetFile(file_path)

# 获取行数
row_count = parquet_file.metadata.num_rows

print(f"文件共有 {row_count} 行")